In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parents[0]   # repo root
sys.path.insert(0, str(ROOT))
##
import pandas as pd 
import numpy as np

from src.features.feature_engineering import add_time_and_lag_features
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.base import clone
from sklearn.linear_model import Ridge

import webbrowser
import time

from src.data.build_clean_df import load_stations_data
from src.optimization.clustering import cluster_candidate_station
from src.optimization.bike_rebalancing import bike_rebalancing
from src.optimization.visualize_opt  import visualize_cluster, print_truck_routes_report, visualize_truck_routes

## Set hour to optimize

In [ ]:
hour_to_optimize = '2017-11-14 08:00:00'

open_html_in_web_browser = False

### Loading processed data

In [3]:
# if this cell is not working please run the models.ipynb notebook to generate the processed data csv
demand = pd.read_csv("../data/processed/divvy_hourly_demand_weather.csv")
demand["hour"] = pd.to_datetime(demand["hour"])
demand = add_time_and_lag_features(demand=demand,lags=(1, 24, 168),drop_na_lags=True)
demand.head()

,station_id,hour,departures,arrivals,net_demand,temp,tmin,tmax,rhum,prcp,...,dow_sin,dow_cos,month_sin,month_cos,dep_lag_1,arr_lag_1,dep_lag_24,arr_lag_24,dep_lag_168,arr_lag_168
168,2,2017-01-08 00:00:00,0,0,0,-11.5,-15.8,-6.8,50,0.0,...,-0.781831,0.62349,0.5,0.866025,0.0,1.0,0.0,0.0,0.0,0.0
169,2,2017-01-08 01:00:00,0,0,0,-11.5,-15.8,-6.8,50,0.0,...,-0.781831,0.62349,0.5,0.866025,0.0,0.0,0.0,0.0,0.0,0.0
170,2,2017-01-08 02:00:00,0,0,0,-11.5,-15.8,-6.8,50,0.0,...,-0.781831,0.62349,0.5,0.866025,0.0,0.0,0.0,0.0,0.0,0.0
171,2,2017-01-08 03:00:00,0,0,0,-11.5,-15.8,-6.8,50,0.0,...,-0.781831,0.62349,0.5,0.866025,0.0,0.0,0.0,0.0,0.0,0.0
172,2,2017-01-08 04:00:00,0,0,0,-11.5,-15.8,-6.8,50,0.0,...,-0.781831,0.62349,0.5,0.866025,0.0,0.0,0.0,0.0,0.0,0.0


In [4]:
stations = load_stations_data()

### Demand prediction

In [5]:
y_dep = demand["departures"]
y_arr = demand["arrivals"]

feature_cols = [
    "hour_sin","hour_cos",
    "dow_sin","dow_cos",
    "month_sin","month_cos",
    "is_weekend",
    "temp","tmin","tmax","rhum","prcp","snwd","wspd","pres",
    "dep_lag_1","dep_lag_24", 'dep_lag_168',
    "arr_lag_1","arr_lag_24", "arr_lag_168"
]

X = demand[feature_cols]

split_time = demand["hour"].quantile(0.8)

train = demand["hour"] <= split_time
test  = demand["hour"] > split_time

X_train = X[train]
X_test  = X[test]

y_dep_train = y_dep[train]
y_dep_test  = y_dep[test]

y_arr_train = y_arr[train]
y_arr_test  = y_arr[test]

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

In [6]:
model = Ridge(alpha=1.0, random_state=42)

print(f"Training Ridge...")

# departures
mdl_dep = clone(model)
mdl_dep.fit(X_train, y_dep_train)
pred_dep = mdl_dep.predict(X_test)

# arrivals
mdl_arr = clone(model)
mdl_arr.fit(X_train, y_arr_train)
pred_arr = mdl_arr.predict(X_test)

pred_dep = np.clip(pred_dep, 0, None)
pred_arr = np.clip(pred_arr, 0, None)

pred_net = pred_dep - pred_arr
true_net = y_dep_test - y_arr_test

results = pd.DataFrame([{
    "model": "Ridge",
    "dep_MAE": mean_absolute_error(y_dep_test, pred_dep),
    "dep_RMSE": rmse(y_dep_test, pred_dep),
    "arr_MAE": mean_absolute_error(y_arr_test, pred_arr),
    "arr_RMSE": rmse(y_arr_test, pred_arr),
    "net_MAE": mean_absolute_error(true_net, pred_net),
    "net_RMSE": rmse(true_net, pred_net),
}])
results.head()

Training Ridge...


,model,dep_MAE,dep_RMSE,arr_MAE,arr_RMSE,net_MAE,net_RMSE
0,Ridge,0.374209,0.945263,0.373062,0.969262,0.456264,1.099768


In [7]:
# build pred_df 
pred_df = demand[test][['station_id','hour','departures','arrivals','net_demand']].rename(columns={
    "net_demand" : "true_net_demand",
    "departures" : "true_departures",
    "arrivals" : "true_arrivals"
})
pred_df['pred_arrivals'] = pred_arr.round()
pred_df['pred_departures'] = pred_dep.round()
pred_df['pred_net_demand'] = pred_net.round()

pred_df = pred_df.merge(stations, left_on='station_id', right_on='id', how='left')
pred_df.drop(columns=['id'], inplace=True)
pred_df = pred_df[['hour','station_id','pred_departures', 'true_departures','pred_arrivals', 'true_arrivals',
       'pred_net_demand','true_net_demand','dpcapacity', 'latitude', 'longitude','name']]

# hourly_net_demand = pred_df.groupby(pred_df['hour'])[['pred_net_demand','true_net_demand']].agg(lambda x: x.abs().sum()).reset_index().sort_values(['pred_net_demand'], ascending=False)
# hourly_net_demand[hourly_net_demand['hour']==hour_to_optimize]



## Filtering candidate and clustering

In [8]:
# function to apply first layer (candidate selection) + 2nd layer (clusterization)
candidate_stations = cluster_candidate_station(pred_df=pred_df, stations=stations, hour_to_optimize=hour_to_optimize,acceptable_margin=0.2, stations_per_cluster=25, random_init_state=True)

In [9]:
# visualize cluster for this hour

visualize_cluster(candidate_stations=candidate_stations, hour_to_optimize=hour_to_optimize)

if open_html_in_web_browser==True :
    webbrowser.open(f"..\plots\clusters_map\station_clusters_{hour_to_optimize[:-6]}.html")

station_clusters_2017-11-14 08.html saved in plots/clusters_map


## bike rebalancing

### testing with only one cluster

In [10]:
cluster_0 = bike_rebalancing(stations=stations,
                     pred_df=pred_df,
                     hour_to_optimize=hour_to_optimize,
                     candidate_stations=candidate_stations,
                     cluster_nb=0,
                     gamma=10,
                     nb_trucks=1,
                     random_init_state=True)
    
cluster_0['objective']

Set parameter Username
Set parameter LicenseID to value 2718992
Academic license - for non-commercial use only - expires 2026-10-07
Solver status: Optimal


19.741249532217548

In [11]:
# print truck report for this cluster
solution = {}
solution[0] = cluster_0
print_truck_routes_report(solutions_dict=solution)


===== TRUCK ROUTE REPORT (ALL CLUSTERS) =====


Truck 0 (cluster 0) :
  Goes from 0 -> 214 | Distance: 0.33 km | Drop: 0, Pick: 8, Truck load before arrival: 0
  Goes from 214 -> 215 | Distance: 1.11 km | Drop: 3, Pick: 0, Truck load before arrival: 8
  Goes from 215 -> 146 | Distance: 1.14 km | Drop: 1, Pick: 0, Truck load before arrival: 5
  Goes from 146 -> 134 | Distance: 1.02 km | Drop: 4, Pick: 0, Truck load before arrival: 5
  Goes from 134 -> 88 | Distance: 0.92 km | Drop: 0, Pick: 12, Truck load before arrival: 1
  Goes from 88 -> 54 | Distance: 1.39 km | Drop: 3, Pick: 0, Truck load before arrival: 13
  Goes from 54 -> 111 | Distance: 1.31 km | Drop: 0, Pick: 15, Truck load before arrival: 10
  Goes from 111 -> 291 | Distance: 1.37 km | Drop: 3, Pick: 0, Truck load before arrival: 25
  Goes from 291 -> 268 | Distance: 0.87 km | Drop: 1, Pick: 0, Truck load before arrival: 23
  Goes from 268 -> 143 | Distance: 1.53 km | Drop: 3, Pick: 0, Truck load before arrival: 22
  Goes f

### testing with all cluster

In [12]:
# loop to optimize within each cluster and collect solutions

solutions_dict = {}
nb_clusters = candidate_stations['cluster'].max() + 1
for cluster in range(nb_clusters):
    print(f'Optimizing cluster {cluster}...')
    start_time = time.time()
    solutions_dict[cluster] = bike_rebalancing(stations=stations,
                     pred_df=pred_df,
                     hour_to_optimize=hour_to_optimize,
                     candidate_stations=candidate_stations,
                     cluster_nb=cluster,
                     gamma=10,
                     nb_trucks=1,
                     random_init_state=True)
    end_time = time.time()
    duration = round(end_time - start_time)
    print(f'Solution found for cluster {cluster} in {duration} sec')


Optimizing cluster 0...
Solver status: Optimal
Solution found for cluster 0 in 4 sec
Optimizing cluster 1...
Solver status: Time limit reached
Solution found for cluster 1 in 60 sec
Optimizing cluster 2...
Solver status: Time limit reached
Solution found for cluster 2 in 61 sec
Optimizing cluster 3...
Solver status: Optimal
Solution found for cluster 3 in 1 sec
Optimizing cluster 4...
Solver status: Optimal
Solution found for cluster 4 in 2 sec
Optimizing cluster 5...
Solver status: Optimal
Solution found for cluster 5 in 7 sec
Optimizing cluster 6...
Solver status: Optimal
Solution found for cluster 6 in 3 sec
Optimizing cluster 7...
Solver status: Optimal
Solution found for cluster 7 in 11 sec


In [13]:
print_truck_routes_report(solutions_dict)


===== TRUCK ROUTE REPORT (ALL CLUSTERS) =====


Truck 0 (cluster 0) :
  Goes from 0 -> 214 | Distance: 0.33 km | Drop: 0, Pick: 8, Truck load before arrival: 0
  Goes from 214 -> 215 | Distance: 1.11 km | Drop: 3, Pick: 0, Truck load before arrival: 8
  Goes from 215 -> 146 | Distance: 1.14 km | Drop: 1, Pick: 0, Truck load before arrival: 5
  Goes from 146 -> 134 | Distance: 1.02 km | Drop: 4, Pick: 0, Truck load before arrival: 5
  Goes from 134 -> 88 | Distance: 0.92 km | Drop: 0, Pick: 12, Truck load before arrival: 1
  Goes from 88 -> 54 | Distance: 1.39 km | Drop: 3, Pick: 0, Truck load before arrival: 13
  Goes from 54 -> 111 | Distance: 1.31 km | Drop: 0, Pick: 15, Truck load before arrival: 10
  Goes from 111 -> 291 | Distance: 1.37 km | Drop: 3, Pick: 0, Truck load before arrival: 25
  Goes from 291 -> 268 | Distance: 0.87 km | Drop: 1, Pick: 0, Truck load before arrival: 23
  Goes from 268 -> 143 | Distance: 1.53 km | Drop: 3, Pick: 0, Truck load before arrival: 22
  Goes f

In [14]:
# visualizing truck routes
visualize_truck_routes(stations=stations,solutions_dict=solutions_dict,hour_to_optimize=hour_to_optimize)
if open_html_in_web_browser==True :
    webbrowser.open(f"..\plots\\truck_routes_map\\truck_routes_map_{hour_to_optimize[:-6]}.html")

truck_routes_map_2017-11-14 08.html saved in plots/truck_routes_map
